# Chapter 05. PyTorch Fundamentals

Chapter 05의 `README.md`에서 개념, 수식, 프로젝트별 실행 조건을 확인할 수 있습니다. 이 Notebook은 데이터 파일 업로드 없이 핵심 연산을 순서대로 실행할 수 있도록 구성했습니다.

기본 실행 환경은 Google Colab의 CPU 런타임입니다. PyTorch가 설치되지 않은 로컬 환경에서는 공식 설치 안내에 따라 설치한 뒤 실행합니다.

## 1. 실행 환경 및 Tensor 생성

Tensor의 **Shape**, **Dtype**, **Device** 속성을 확인합니다.

In [ ]:
import torch
import numpy as np
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]], dtype=torch.float32)
print('shape:', x.shape, 'dtype:', x.dtype, 'device:', x.device)
assert x.shape == (2, 3)

## 2. Indexing과 Shape 변환

행·열 선택과 `reshape`, `unsqueeze`의 차이를 확인합니다.

In [ ]:
print('first row:', x[0].tolist())
print('second column:', x[:, 1].tolist())
print('flat:', x.reshape(-1).tolist())
print('with batch axis:', tuple(x.unsqueeze(0).shape))
assert tuple(x.unsqueeze(0).shape) == (1, 2, 3)

## 3. Broadcasting, Mask, Reduction

센서별 보정값을 한 번에 더하고, 결측값 `-1`을 제외한 평균을 계산합니다.

In [ ]:
readings = torch.tensor([[12., -1., 19.], [14., 15., -1.]])
offsets = torch.tensor([1., 2., 3.])
valid = readings >= 0
corrected = readings + offsets
counts = valid.sum(dim=0)
mean_values = torch.where(valid, corrected, torch.zeros_like(corrected)).sum(dim=0) / counts
print('corrected:', corrected.tolist())
print('counts:', counts.tolist())
print('means:', mean_values.tolist())
assert counts.tolist() == [2, 1, 1]
assert mean_values.tolist() == [14., 17., 22.]

## 4. Matrix Multiplication

원소별 곱과 행렬 곱은 다릅니다. 행렬 곱은 안쪽 축의 크기가 같아야 합니다.

In [ ]:
features = torch.tensor([[1., 2.], [3., 4.]])
weights = torch.tensor([[10.], [100.]])
scores = features @ weights
print('scores:', scores.tolist(), 'shape:', tuple(scores.shape))
assert scores.tolist() == [[210.], [430.]]

## 5. NumPy와 Tensor 메모리 공유

`from_numpy()`는 일반적으로 NumPy 배열과 메모리를 공유합니다.

In [ ]:
source = np.array([1., 2., 3.], dtype=np.float32)
shared = torch.from_numpy(source)
copied = torch.tensor(source)
source[0] = 50.
print('shared:', shared.tolist())
print('copied:', copied.tolist())
assert shared[0].item() == 50. and copied[0].item() == 1.

## 6. CPU/GPU Device 선택

CUDA와 MPS가 없으면 CPU에서 실행합니다.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
a = torch.arange(5, dtype=torch.float32, device=device)
print('device:', device, 'result:', (a * 2).detach().cpu().tolist())

## 7. Autograd와 Chain Rule

`L=(wx+b)^2`에서 `w=2`, `x=3`, `b=1`일 때 `dL/dw=42`, `dL/db=14`입니다.

In [ ]:
w = torch.tensor(2., requires_grad=True)
b = torch.tensor(1., requires_grad=True)
x_value = torch.tensor(3.)
loss = (w * x_value + b).square()
loss.backward()
print('loss:', loss.item(), 'dw:', w.grad.item(), 'db:', b.grad.item())
assert w.grad.item() == 42.
assert b.grad.item() == 14.

## 8. Gradient 누적과 초기화

새 그래프로 `.backward()`를 호출하면 기존 Gradient에 값이 더해집니다.

In [ ]:
p = torch.tensor(2.0, requires_grad=True)
(p * 3).backward()
first = p.grad.item()
(p * 3).backward()
second = p.grad.item()
p.grad = None
print('first:', first, 'accumulated:', second, 'reset:', p.grad)
assert (first, second) == (3., 6.)

## 9. 한 단계 Gradient Descent

Autograd로 계산한 Gradient를 이용해 한 단계만 Parameter를 갱신합니다.

In [ ]:
observed_x = torch.tensor([1., 2., 3.])
observed_y = torch.tensor([2., 4., 6.])
w = torch.tensor(0., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
loss_before = ((w * observed_x + b - observed_y) ** 2).mean()
loss_before.backward()
with torch.no_grad():
    w -= 0.05 * w.grad
    b -= 0.05 * b.grad
    loss_after = ((w * observed_x + b - observed_y) ** 2).mean()
print('loss before:', loss_before.item(), 'after:', loss_after.item())
assert loss_after.item() < loss_before.item()
w.grad = None
b.grad = None

## 10. TensorDataset과 DataLoader

10개 Sample을 4개씩 나누면 마지막 Batch에는 2개가 남습니다.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
X = torch.arange(30, dtype=torch.float32).reshape(10, 3)
labels = (X[:, 0] > 12).long()
loader = DataLoader(TensorDataset(X, labels), batch_size=4, shuffle=False, num_workers=0)
batch_sizes = [len(features) for features, batch_labels in loader]
print('batch sizes:', batch_sizes)
assert batch_sizes == [4, 4, 2]

## 11. Custom Dataset

`__len__`과 `__getitem__`을 구현한 뒤 DataLoader와 연결합니다.

In [ ]:
from torch.utils.data import Dataset
class SimpleDataset(Dataset):
    def __init__(self):
        self.inputs = torch.arange(12, dtype=torch.float32).reshape(4, 3)
        self.targets = torch.tensor([0, 1, 0, 1], dtype=torch.int64)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

custom = SimpleDataset()
for batch_x, batch_y in DataLoader(custom, batch_size=3, shuffle=False):
    print('features:', tuple(batch_x.shape), 'targets:', batch_y.tolist())
assert len(custom) == 4

## 다음 실습

- **Mini 01:** 센서 CSV의 보정·Mask·집계
- **Mini 02:** Autograd를 사용한 보정식 Parameter 갱신
- **Advanced 01:** 실제 CSV를 읽는 Custom Dataset과 DataLoader
- **Advanced 02:** Autograd와 수치 미분의 Gradient 비교

각 프로젝트의 입력 데이터, 구현 조건, 테스트 시나리오는 Chapter 05의 `README.md`와 `projects/` 폴더에서 확인할 수 있습니다.